#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=100,chunk_overlap=30)
docs=text_splitter.split_documents(documents)

C:\Users\shiva\AppData\Local\Temp\ipykernel_12228\2884781553.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
Created a chunk of size 207, which is longer than the specified 100


In [2]:
print(len(docs))

5


In [3]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='Hello everyone! Welcome to my Generative AI project.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Today, I am learning how artificial intelligence can understand human language and generate meaningful responses. Generative AI can help us write content, answer questions, create images, and solve problems.'),
 Document(metadata={'source': 'speech.txt'}, page_content='I am also exploring Python, LangChain, and Google Gemini to build intelligent applications.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Learning something new takes time and practice, but every small step brings us closer to our goals.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Thank you for listening!')]

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

db = FAISS.from_documents(docs, embeddings)

C:\Users\shiva\AppData\Local\Temp\ipykernel_12228\1762571002.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [5]:
query = "Which tools am I exploring to build intelligent applications?"

docs = db.similarity_search(query)

print(docs[0].page_content)

I am also exploring Python, LangChain, and Google Gemini to build intelligent applications.


As a Retriever

We can also convert vectorstore to retriver class.This allows us to easily use it in other Langchain methods,which largely work with retrievers.
It acts like a interface which connects with vectorstore and retrieve the details from the vectorstore and provides the response.

In [6]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'I am also exploring Python, LangChain, and Google Gemini to build intelligent applications.'

Similarity Search

There are some FAISS specific methods.One of them is similarity_search_with_score which allows you to return not only documents but also distance_score of the query to them.The rreturned distance score is L2 distance.Therfore,a lower score is better.

In [7]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='05b46ffb-6954-4336-957d-53c02b268e50', metadata={'source': 'speech.txt'}, page_content='I am also exploring Python, LangChain, and Google Gemini to build intelligent applications.'),
  np.float32(313.03918)),
 (Document(id='3c94c769-6a65-4d84-bcd3-0855b60bef7b', metadata={'source': 'speech.txt'}, page_content='Today, I am learning how artificial intelligence can understand human language and generate meaningful responses. Generative AI can help us write content, answer questions, create images, and solve problems.'),
  np.float32(381.89648)),
 (Document(id='95106f6a-ac7c-4681-9c0f-0b998af3cee2', metadata={'source': 'speech.txt'}, page_content='Hello everyone! Welcome to my Generative AI project.'),
  np.float32(420.64044)),
 (Document(id='68941b15-b785-4a1f-a33d-d894fa601212', metadata={'source': 'speech.txt'}, page_content='Learning something new takes time and practice, but every small step brings us closer to our goals.'),
  np.float32(463.8853))]

In [10]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.30586302280426025,
 1.1358284950256348,
 -3.145753860473633,
 0.09992843866348267,
 0.9745128154754639,
 -0.9505747556686401,
 0.14814595878124237,
 -0.281953364610672,
 -0.36766451597213745,
 -1.0768390893936157,
 0.36775052547454834,
 -0.23681914806365967,
 2.2325801849365234,
 0.8017079830169678,
 -0.7244567275047302,
 0.23238910734653473,
 0.5467357635498047,
 -1.0774970054626465,
 0.9077820777893066,
 0.06206673011183739,
 0.43240785598754883,
 -1.4910976886749268,
 0.4643462896347046,
 -0.6794688701629639,
 1.13407301902771,
 0.25435203313827515,
 -0.5934114456176758,
 0.37601912021636963,
 -0.020453810691833496,
 1.0666741132736206,
 0.932572603225708,
 -0.7060784697532654,
 -0.6724556684494019,
 0.7310676574707031,
 -0.413796067237854,
 -0.7795417308807373,
 1.1165237426757812,
 -0.08249034732580185,
 -0.6096253395080566,
 0.9585492610931396,
 -0.4356032907962799,
 -0.40048229694366455,
 0.6013393998146057,
 -0.853534460067749,
 1.8382835388183594,
 0.2755274176597595,
 1.6

In [11]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='05b46ffb-6954-4336-957d-53c02b268e50', metadata={'source': 'speech.txt'}, page_content='I am also exploring Python, LangChain, and Google Gemini to build intelligent applications.'),
 Document(id='3c94c769-6a65-4d84-bcd3-0855b60bef7b', metadata={'source': 'speech.txt'}, page_content='Today, I am learning how artificial intelligence can understand human language and generate meaningful responses. Generative AI can help us write content, answer questions, create images, and solve problems.'),
 Document(id='95106f6a-ac7c-4681-9c0f-0b998af3cee2', metadata={'source': 'speech.txt'}, page_content='Hello everyone! Welcome to my Generative AI project.'),
 Document(id='68941b15-b785-4a1f-a33d-d894fa601212', metadata={'source': 'speech.txt'}, page_content='Learning something new takes time and practice, but every small step brings us closer to our goals.')]

In [12]:
##save and loading
db.save_local("faiss_index")

In [ ]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)